# M4E1 GSE79275 Arabidopsis Data Analysis 

In this exercise, you will perform a differential expression analysis on the [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275) data set. 

There are no requirements as to which graphs to make specifically; however, you must end up with your DEG results, check them for quality, and be able to interpret and defend the results for the discussion. These results should coincide with our expected result, as according to [this paper](https://pubmed.ncbi.nlm.nih.gov/34499172/):

**In *Arabidopsis thaliana*, excess cadmium (Cd) primarily disrupts iron (Fe) homeostasis by inducing a physiological Fe deficiency in the shoots while causing Fe accumulation in the roots. This disruption is a key mechanism of Cd toxicity and involves several specific interactions with the plant's iron uptake and transport systems**

The samples provided are 6 root samples for Col-0, Cd-treated, and untreated (replete). These samples are averaged from 3 technical replicates each. If you would like, you are welcome to compare the roots and the shoots together too. The counts and metadata are loaded below. The samples loaded are listed in a table format as well: 

| sample_info           | genotype | tissue | treatment           | bio_rep |
| --------------------- | -------- | ------ | ------------------- | ------- |
| col0_Root_Untreated_1 | col0     | Root   | replete             | 1       |
| col0_Root_Untreated_2 | col0     | Root   | replete             | 2       |
| col0_Root_Untreated_3 | col0     | Root   | replete             | 3       |
| col0_Root_Treated_1   | col0     | Root   | 72hr CD application | 1       |
| col0_Root_Treated_2   | col0     | Root   | 72hr CD application | 2       |
| col0_Root_Treated_3   | col0     | Root   | 72hr CD application | 3       |


![image.png](images/M4E1_GSE79275_Arabidopsis_Data_Analysis_c804d178.png)


In [ ]:
### load libraries
library(dplyr)
library(tidyr)
library(tibble)
library(ggplot2)
library(reshape2)
library(stringr)
library(DESeq2)
library(tximport)
library(clusterProfiler)
library(org.At.tair.db)
library(biomaRt)
library(pathview)
library(GO.db)
library(AnnotationDbi)

In [ ]:
DATA_DIR <- Sys.getenv("DATA_DIR", unset = "../../../data")  # repository data/ folder; run data/fetch_data.sh geo first
counts <- read.delim(file.path(DATA_DIR, "geo", "GSE79275", "GSE79275_raw_read_counts.txt"), header=TRUE, stringsAsFactors=FALSE)

rownames(counts) <- counts$AGI

counts <- round(as.matrix(counts[,-1]))

sample_names <- colnames(counts)

genotype <- ifelse(grepl("^C", sample_names), "col0", "opt3")

tissue <- ifelse(grepl("R", sample_names), "Root", "Shoot")

treatment <- ifelse(grepl("T", sample_names), "Treated", "Untreated")

bio_rep <- as.integer(sub(".*?([0-9]+)[-_][0-9]+\\.bam$", "\\1", sample_names))

colData <- data.frame(sample=sample_names, genotype=factor(genotype), tissue=factor(tissue),
                      treatment=factor(treatment), bio_rep=bio_rep, stringsAsFactors=FALSE)

colData <- colData[order(colData$bio_rep, colData$sample),]

colData$tech_rep <- ave(colData$bio_rep, colData$bio_rep, FUN=seq_along)

rownames(colData) <- colData$sample

colData$sample <- NULL

meta <- colData

meta <- meta[colnames(counts),]

all(colnames(counts) %in% rownames(meta))

all(colnames(counts) == rownames(meta))

meta$bio_id <- paste(meta$genotype, meta$tissue, meta$treatment, meta$bio_rep, sep="_")

collapsed_counts <- sapply(unique(meta$bio_id), function(id) rowSums(counts[, meta$bio_id==id]))

collapsed_counts <- as.matrix(collapsed_counts)

collapsed_meta <- meta %>% distinct(bio_id, .keep_all=TRUE) %>% dplyr::select(-tech_rep)

rownames(collapsed_meta) <- collapsed_meta$bio_id

col0_root_meta <- collapsed_meta %>% filter(tissue=="Root", genotype=="col0")

col0_root_counts <- collapsed_counts[, rownames(col0_root_meta)]

col0_root_meta$treatment <- relevel(col0_root_meta$treatment, ref="Untreated")

col0_root_meta
col0_root_counts

In [ ]:
#your code here